In [1]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [3]:
df = pd.read_csv('HIVDR Integrated Dataset.csv', low_memory=False)

# Data Cleaning
# Removing observations with missing values in the 'HIVDR_presence' column
print("HIVDR_presence categories and counts:")
print(df['HIVDR_presence'].value_counts())
print()
missing_count = df['HIVDR_presence'].isna().sum() + (df['HIVDR_presence'] == 'Missing').sum()
print(f"There are {missing_count} missing values")
df = df.dropna(subset=['HIVDR_presence'])
df = df[df['HIVDR_presence'] != 'Missing']
remaining_missing = df['HIVDR_presence'].isna().sum() + (df['HIVDR_presence'] == 'Missing').sum()
print(f"After removal: {remaining_missing} missing values remain")
print()

# Identifying and removing duplicate rows, keeping the first occurrence
print(f"Number of duplicate rows: {df['personid'].duplicated().sum()}")
df = df.drop_duplicates(subset='personid', keep='first')
print(f"Number of rows after removing duplicates: {len(df)}")
print()

# Dropping all 144 survey weight columns
df = df.drop(columns=df.loc[:, 'intwt0':'intwt141'].columns)
rows, cols = df.shape
print(f"After removing survey weights, the dataset has {rows} rows and {cols} columns.")
print()

# Missing value recoding except 'known_hiv_status'
cols_to_replace = [col for col in df.columns if col != 'known_hiv_status']
df[cols_to_replace] = df[cols_to_replace].replace(99, np.nan)

# Removing features with ≥30% missing values
df = df.dropna(thresh=0.7*len(df), axis=1)
rows, cols = df.shape
print(f"After removing columns with more than 30% missing values, the dataset has {rows} rows and {cols} columns.")
print()

# Missing data imputation
#Columns excluded from imputation
exclude_from_imputation = ['householdid', 'personid', 'subtype', 'HIVDR_presence', 'centroidid', 'surveystyear', 'surveystmonth']
# Defining categorical and continuous columns
continuous_cols = ['age', 'firstsxage', 'wealthscorecont']
categorical_cols = [col for col in df.columns 
                   if col not in continuous_cols + exclude_from_imputation]
# Checking which column types have missing values
cat_cols_with_missing = [col for col in categorical_cols 
                         if df[col].isnull().any()]
cont_cols_with_missing = [col for col in continuous_cols 
                          if df[col].isnull().any()]
print(f"Categorical columns with missing values: {cat_cols_with_missing}")
print(f"Continuous columns with missing values: {cont_cols_with_missing}")
# Imputing continuous variables with MICE
if cont_cols_with_missing:
    mice_imputer = IterativeImputer(random_state=42, max_iter=10, verbose=1)
    df[cont_cols_with_missing] = mice_imputer.fit_transform(df[cont_cols_with_missing])
    print(f"Imputed continuous columns: {cont_cols_with_missing}")
# Imputing categorical variables with mode
if cat_cols_with_missing:
    for col in cat_cols_with_missing:
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)
    print(f"Imputed categorical columns: {cat_cols_with_missing}")
imputed_cols = [col for col in df.columns if col not in exclude_from_imputation]
print(f"\nMissing values after imputation: {df[imputed_cols].isnull().sum().sum()}")
print()

# Removing features where ≥90% of values are identical
threshold = 0.90
columns_to_drop = [col for col in df.columns 
                   if df[col].value_counts(normalize=True).iloc[0] >= threshold]
print(f"Columns to drop (>={threshold*100}% same value): {len(columns_to_drop)}")
print(f"Column names: {columns_to_drop}")
df = df.drop(columns=columns_to_drop)
rows, cols = df.shape
print(f"After removing columns where ≥90% of values are the same, the dataset has {rows} rows and {cols} columns.")
print()

#Removing indeterminate and non-informative features
non_informative_columns = ['householdid', 'personid', 'centroidid', 'surveystyear', 'surveystmonth', 'linenumfixed', 'hivtstlocation', 'relattohh', 'hivtstever', 'hivtstrslt', 'subtype', 'wealthscorecont', 'agegroup5population', 'tbclinvisit', 'partagain1', 'evermar', 'schcom', 'sick3mo', 'firstsxage', 'partrelation1','known_hiv_status', 'testedreceiveddetail', 'allpartnersspouselivein12months', 'tri90aware', 'partlastcndm1', 'avoidpreg', 'partlivew1', 'uniontype', 'evertested']
df = df.drop(columns=non_informative_columns, errors='ignore')
rows, cols = df.shape
print(f"After removing non-informative features, the dataset has {rows} rows and {cols} columns.")
print()

# Categorical variable decoding
# 1. cd4cat - CD4 category (cells/μL)
cd4cat_labels = {
    1.0: 'CD4 < 200',
    2.0: 'CD4 < 200',
    3.0: '200 ≤ CD4 < 350',
    4.0: '350 ≤ CD4 < 500',
    5.0: 'CD4 ≥ 500'
}
df['cd4cat'] = df['cd4cat'].map(cd4cat_labels)

# 2. recentlagvl - LAg: recent/long term infection, final, code
recentlagvl_labels = {
    1.0: 'Positive: recent',
    2.0: 'Positive: long term'
}
df['recentlagvl'] = df['recentlagvl'].map(recentlagvl_labels)

# 3. recentlagvlarv - LAg: recent/long term infection, final, code; considers laboratory ARV results
recentlagvlarv_labels = {
    1.0: 'Positive: recent',
    2.0: 'Positive: long term'
}
df['recentlagvlarv'] = df['recentlagvlarv'].map(recentlagvlarv_labels)

# 4. arvefv - Results of test for Efavirenz in blood
arvefv_labels = {
    1.0: 'Positive',
    2.0: 'Negative'
}
df['arvefv'] = df['arvefv'].map(arvefv_labels)

# 5. artduration - Duration of time on ART
artduration_labels = {
    1.0: 'On ART ≥ 24 months',
    2.0: 'On ART 12-23 months',
    3.0: 'On ART < 12 months',
    4.0: 'Not on ART'
}
df['artduration'] = df['artduration'].map(artduration_labels)

# 6. arvstatus - Indicator of whether ARVs detected
arvstatus_labels = {
    1.0: 'ARVs detected',
    2.0: 'ARVs not detected'
}
df['arvstatus'] = df['arvstatus'].map(arvstatus_labels)

# 7. aware - Awareness combining self-report and ARV testing
aware_labels = {
    1.0: 'Aware (ARVs detected)',
    2.0: 'Unaware (ARVs not detected/unknown)'
}
df['aware'] = df['aware'].map(aware_labels)

# 8. awareselfreported - Indicator of whether individual is self-reported as aware of their HIV seropositive status
awareselfreported_labels = {
    1.0: 'Self-report aware of HIV+ status',
    2.0: 'Self-report not aware of HIV+ status'
}
df['awareselfreported'] = df['awareselfreported'].map(awareselfreported_labels)

# 9. awareartselfreported - Determine if patient self-reported as previously diagnosed and if they were on ART
awareartselfreported_labels = {
    1.0: 'Self-report not previously diagnosed',
    2.0: 'Self-report previously diagnosed, not on ART',
    3.0: 'Self-report previously diagnosed, on ART'
}
df['awareartselfreported'] = df['awareartselfreported'].map(awareartselfreported_labels)

# 10. gender - gender
gender_labels = {
    1.0: 'Male',
    2.0: 'Female'
}
df['gender'] = df['gender'].map(gender_labels)

# 11. condomlastsex12months - Indicator for used condom at last sexual encounter in the past 12 months
condomlastsex12months_labels = {
    1.0: 'Condom used',
    2.0: 'Condom not used',
    3.0: 'No sex'
}
df['condomlastsex12months'] = df['condomlastsex12months'].map(condomlastsex12months_labels)

# 12. sex12months - Flag if respondent had sexual intercourse in the past 12 months
sex12months_labels = {
    1.0: 'Had sex',
    2.0: 'No sex'
}
df['sex12months'] = df['sex12months'].map(sex12months_labels)

# 13. urban - Urban Area Indicator (rename to residenceType)
# First, rename the column
df = df.rename(columns={'urban': 'residenceType'})

# Standard mapping for most countries
standard_labels = {
    1.0: 'Urban',
    2.0: 'Rural'
}

# Lesotho-specific mapping
lesotho_labels = {
    1.0: 'Urban',
    2.0: 'Urban',
    3.0: 'Rural'
}

# Apply mapping based on country
df['residenceType'] = df['residenceType'].astype('object')
df.loc[df['country'] == 'Lesotho', 'residenceType'] = df.loc[df['country'] == 'Lesotho', 'residenceType'].map(lesotho_labels)
df.loc[df['country'] != 'Lesotho', 'residenceType'] = df.loc[df['country'] != 'Lesotho', 'residenceType'].map(standard_labels)

# 14. parthivsat1 - What is the HIV status of [PARTNER'S NAME]?
parthivsat1_labels = {
    1.0: 'Positive',
    2.0: 'Positive',
    3.0: 'Positive',
    123.0: 'Positive',
    4.0: 'Negative',
    5.0: 'Negative',
    6.0: 'Negative',
    7.0: "Don't know",
    -8.0: "Don't know",
    -9.0: "Don't know"
}
df['parthivsat1'] = df['parthivsat1'].map(parthivsat1_labels)

# 15. wealthquintile - Wealth quintile
wealthquintile_labels = {
    1.0: 'Lowest',
    2.0: 'Second',
    3.0: 'Middle',
    4.0: 'Fourth',
    5.0: 'Highest'
}
df['wealthquintile'] = df['wealthquintile'].map(wealthquintile_labels)

# 16. partknowhiv1 - Does [PARTNER'S NAME] know your HIV status?
partknowhiv1_labels = {
    1.0: 'Yes',
    2.0: 'No',
    -8.0: 'No',
    -9.0: 'No'
}
df['partknowhiv1'] = df['partknowhiv1'].map(partknowhiv1_labels)

# 17. curmar - What is your marital status now
curmar_labels = {
    1.0: 'Married',
    2.0: 'Living together',
    3.0: 'Widowed',
    4.0: 'Divorced/separated',
    5.0: 'Divorced/separated'
}
df['curmar'] = df['curmar'].map(curmar_labels)

# 18. work12mo - Have you done any work in the last 12 months for which you received cash or goods as payment?
work12mo_labels = {
    1.0: 'Yes',
    2.0: 'No'
}
df['work12mo'] = df['work12mo'].map(work12mo_labels)

# 19. schlat - Have you ever attended school?
schlat_labels = {
    1.0: 'Yes',
    2.0: 'No'
}
df['schlat'] = df['schlat'].map(schlat_labels)

print("Label conversion completed successfully!")
print("\nSample of converted data:")
df.head(10)

HIVDR_presence categories and counts:
HIVDR_presence
No         620
Missing    591
Yes        571
Name: count, dtype: int64

There are 591 missing values
After removal: 0 missing values remain

Number of duplicate rows: 0
Number of rows after removing duplicates: 1191

After removing survey weights, the dataset has 1191 rows and 179 columns.

After removing columns with more than 30% missing values, the dataset has 1191 rows and 63 columns.

Categorical columns with missing values: ['cd4cat', 'recentlagvl', 'recentlagvlarv', 'arvefv', 'arvlpv', 'arvnvp', 'artduration', 'arvstatus', 'tri90aware', 'aware', 'awareartselfreported', 'sick3mo', 'schlcur', 'schcom', 'curmar', 'avoidpreg', 'partlivew1', 'partrelation1', 'partlastcndm1', 'partlastsup1', 'partagain1', 'partknowhiv1', 'parthivsat1', 'hivtstlocation', 'hivtstrslt', 'uniontype', 'testedreceiveddetail', 'sex12months', 'sexever', 'condomlastsex12months', 'allpartnersspouselivein12months']
Continuous columns with missing values: ['fir

,country,HIVDR_presence,cd4cat,recentlagvl,recentlagvlarv,arvefv,artduration,arvstatus,aware,awareselfreported,...,age,schlat,work12mo,curmar,partknowhiv1,parthivsat1,residenceType,wealthquintile,sex12months,condomlastsex12months
0,Cameroon,No,CD4 ≥ 500,Positive: long term,Positive: long term,Positive,On ART < 12 months,ARVs detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,37.0,Yes,Yes,Married,No,Don't know,Rural,Middle,Had sex,Condom not used
1,Cameroon,No,CD4 < 200,Positive: long term,Positive: long term,Negative,On ART ≥ 24 months,ARVs not detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,44.0,Yes,Yes,Living together,Yes,Don't know,Urban,Fourth,Had sex,Condom used
2,Cameroon,Yes,350 ≤ CD4 < 500,Positive: long term,Positive: long term,Negative,On ART ≥ 24 months,ARVs not detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,40.0,Yes,Yes,Widowed,Yes,Don't know,Rural,Lowest,No sex,No sex
3,Cameroon,No,CD4 ≥ 500,Positive: long term,Positive: long term,Negative,On ART 12-23 months,ARVs not detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,41.0,Yes,Yes,Living together,No,Don't know,Urban,Fourth,Had sex,Condom not used
4,Cameroon,Yes,350 ≤ CD4 < 500,Positive: long term,Positive: long term,Negative,On ART ≥ 24 months,ARVs detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,32.0,Yes,Yes,Living together,Yes,Positive,Urban,Fourth,Had sex,Condom not used
9,Cameroon,No,CD4 ≥ 500,Positive: long term,Positive: long term,Positive,Not on ART,ARVs detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,37.0,Yes,Yes,Married,No,Negative,Urban,Highest,Had sex,Condom not used
10,Cameroon,Yes,CD4 < 200,Positive: long term,Positive: long term,Negative,On ART ≥ 24 months,ARVs not detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,40.0,Yes,No,Living together,Yes,Negative,Rural,Lowest,Had sex,Condom not used
11,Cameroon,Yes,CD4 < 200,Positive: long term,Positive: long term,Negative,On ART ≥ 24 months,ARVs not detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,30.0,Yes,Yes,Living together,No,Negative,Urban,Fourth,Had sex,Condom not used
12,Cameroon,Yes,350 ≤ CD4 < 500,Positive: recent,Positive: long term,Positive,On ART < 12 months,ARVs detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,57.0,Yes,No,Widowed,Yes,Don't know,Urban,Highest,No sex,No sex
13,Cameroon,Yes,350 ≤ CD4 < 500,Positive: long term,Positive: long term,Negative,On ART < 12 months,ARVs not detected,Aware (ARVs detected),Self-report aware of HIV+ status,...,37.0,Yes,Yes,Married,Yes,Negative,Urban,Fourth,Had sex,Condom used


In [4]:
#Feature engineering
#Converting continuous age to categorical bands
bins = [15, 25, 50, np.inf]
labels = ['15–24 years', '25–49 years', '≥50 years']
df['age'] = pd.cut(df['age'], bins=bins, labels=labels, right=False, ordered=True).astype('category')
print ("Age categories:")
print(df['age'].value_counts())

Age categories:
age
25–49 years    836
15–24 years    224
≥50 years      131
Name: count, dtype: int64


In [5]:
df.to_csv('Chi-square Analysis Data_Final.csv', index=False, encoding='utf-8-sig')